# Так или иначе: фреймворк матчей, Excel и парсинга ставок

Заготовка закрывает три задачи: выбор 12 матчей, генерация Excel-шаблона и перенос текстовых ставок игроков в нормализованный Excel.

## Правила, которые валидируются

- Ровно 5 ставок на банк 5000 у.е.
- Каждая ставка от 500 до 2000 и кратна 50.
- Доступные рынки: `П1`, `П2`, `Х`, `1Х`, `Х2`, `ТМ`, `ТБ`.
- Один матч не повторяется в нескольких ставках игрока.
- Структуры: 4 ординара + 1 экспресс или 3 ординара + 2 экспресса.
- Экспресс состоит из 2 или 3 событий.
- В линии из 12 матчей максимум 1 матч, где коэффициент победы фаворита ниже 1.8.

In [ ]:
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'output'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RULES = {
    'bank': 5000,
    'bets_count': 5,
    'stake_min': 500,
    'stake_max': 2000,
    'stake_step': 50,
    'line_matches': 12,
    'max_low_favorite_matches': 1,
    'low_favorite_threshold': 1.8,
    'allowed_markets': {'П1', 'П2', 'Х', '1Х', 'Х2', 'ТМ', 'ТБ'},
}

RULES

In [ ]:
def load_fixtures() -> pd.DataFrame:
    path = DATA_DIR / 'fixtures_odds.csv'
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame([
        {'match_id': 'm01', 'kickoff': '2026-05-01 20:00', 'league': 'EPL', 'home_team': 'Arsenal', 'away_team': 'Chelsea', 'odds_p1': 1.95, 'odds_x': 3.60, 'odds_p2': 4.10, 'odds_tm': 1.90, 'odds_tb': 1.92},
        {'match_id': 'm02', 'kickoff': '2026-05-01 22:00', 'league': 'La Liga', 'home_team': 'Barcelona', 'away_team': 'Real Madrid', 'odds_p1': 2.40, 'odds_x': 3.30, 'odds_p2': 2.90, 'odds_tm': 2.05, 'odds_tb': 1.78},
        {'match_id': 'm03', 'kickoff': '2026-05-02 17:00', 'league': 'Serie A', 'home_team': 'Inter', 'away_team': 'Milan', 'odds_p1': 1.72, 'odds_x': 3.80, 'odds_p2': 4.80, 'odds_tm': 1.86, 'odds_tb': 1.98},
        {'match_id': 'm04', 'kickoff': '2026-05-02 19:30', 'league': 'Bundesliga', 'home_team': 'Bayern', 'away_team': 'Dortmund', 'odds_p1': 1.82, 'odds_x': 4.00, 'odds_p2': 4.00, 'odds_tm': 2.20, 'odds_tb': 1.65},
        {'match_id': 'm05', 'kickoff': '2026-05-02 21:45', 'league': 'Ligue 1', 'home_team': 'PSG', 'away_team': 'Lyon', 'odds_p1': 1.55, 'odds_x': 4.30, 'odds_p2': 6.00, 'odds_tm': 2.15, 'odds_tb': 1.70},
        {'match_id': 'm06', 'kickoff': '2026-05-03 15:00', 'league': 'EPL', 'home_team': 'Liverpool', 'away_team': 'Tottenham', 'odds_p1': 1.88, 'odds_x': 3.90, 'odds_p2': 4.20, 'odds_tm': 2.05, 'odds_tb': 1.78},
        {'match_id': 'm07', 'kickoff': '2026-05-03 17:15', 'league': 'La Liga', 'home_team': 'Atletico Madrid', 'away_team': 'Sevilla', 'odds_p1': 1.91, 'odds_x': 3.45, 'odds_p2': 4.50, 'odds_tm': 1.72, 'odds_tb': 2.12},
        {'match_id': 'm08', 'kickoff': '2026-05-03 19:30', 'league': 'Eredivisie', 'home_team': 'Ajax', 'away_team': 'PSV', 'odds_p1': 2.55, 'odds_x': 3.70, 'odds_p2': 2.60, 'odds_tm': 2.35, 'odds_tb': 1.60},
        {'match_id': 'm09', 'kickoff': '2026-05-03 21:45', 'league': 'Primeira', 'home_team': 'Benfica', 'away_team': 'Porto', 'odds_p1': 2.30, 'odds_x': 3.20, 'odds_p2': 3.20, 'odds_tm': 1.78, 'odds_tb': 2.04},
        {'match_id': 'm10', 'kickoff': '2026-05-04 18:00', 'league': 'RPL', 'home_team': 'Краснодар', 'away_team': 'Зенит', 'odds_p1': 2.95, 'odds_x': 3.25, 'odds_p2': 2.42, 'odds_tm': 1.84, 'odds_tb': 1.98},
        {'match_id': 'm11', 'kickoff': '2026-05-04 20:30', 'league': 'EPL', 'home_team': 'Manchester City', 'away_team': 'Newcastle', 'odds_p1': 1.68, 'odds_x': 4.10, 'odds_p2': 5.00, 'odds_tm': 2.10, 'odds_tb': 1.72},
        {'match_id': 'm12', 'kickoff': '2026-05-04 22:00', 'league': 'Serie A', 'home_team': 'Napoli', 'away_team': 'Roma', 'odds_p1': 2.15, 'odds_x': 3.35, 'odds_p2': 3.50, 'odds_tm': 1.80, 'odds_tb': 2.02},
        {'match_id': 'm13', 'kickoff': '2026-05-05 20:00', 'league': 'Bundesliga', 'home_team': 'Leipzig', 'away_team': 'Leverkusen', 'odds_p1': 2.60, 'odds_x': 3.55, 'odds_p2': 2.65, 'odds_tm': 2.05, 'odds_tb': 1.78},
        {'match_id': 'm14', 'kickoff': '2026-05-05 22:00', 'league': 'EPL', 'home_team': 'Manchester United', 'away_team': 'Aston Villa', 'odds_p1': 2.05, 'odds_x': 3.45, 'odds_p2': 3.55, 'odds_tm': 1.88, 'odds_tb': 1.94},
    ])


def load_power_rankings() -> pd.DataFrame:
    path = DATA_DIR / 'team_power_rankings.csv'
    if path.exists():
        return pd.read_csv(path)
    teams = ['Arsenal', 'Chelsea', 'Barcelona', 'Real Madrid', 'Inter', 'Milan', 'Bayern', 'Dortmund', 'PSG', 'Lyon', 'Liverpool', 'Tottenham', 'Atletico Madrid', 'Sevilla', 'Ajax', 'PSV', 'Benfica', 'Porto', 'Краснодар', 'Зенит', 'Manchester City', 'Newcastle', 'Napoli', 'Roma', 'Leipzig', 'Leverkusen', 'Manchester United', 'Aston Villa']
    return pd.DataFrame({'team': teams, 'power_score': [94, 86, 93, 95, 92, 86, 94, 88, 93, 82, 94, 85, 89, 80, 82, 86, 84, 85, 78, 81, 97, 86, 88, 84, 86, 90, 84, 82]})


fixtures = load_fixtures()
power = load_power_rankings()
fixtures.head(), power.head()

In [ ]:
def enrich_fixtures(fixtures: pd.DataFrame, power: pd.DataFrame) -> pd.DataFrame:
    home_power = power.rename(columns={'team': 'home_team', 'power_score': 'home_power'})
    away_power = power.rename(columns={'team': 'away_team', 'power_score': 'away_power'})
    out = fixtures.merge(home_power, on='home_team', how='left').merge(away_power, on='away_team', how='left')
    out[['home_power', 'away_power']] = out[['home_power', 'away_power']].fillna(70)
    out['favorite_odds'] = out[['odds_p1', 'odds_p2']].min(axis=1)
    out['low_favorite_flag'] = out['favorite_odds'] < RULES['low_favorite_threshold']
    out['power_sum'] = out['home_power'] + out['away_power']
    out['power_gap_abs'] = (out['home_power'] - out['away_power']).abs()
    out['odds_balance'] = 1 / (1 + out['power_gap_abs'])
    out['match_score'] = 0.65 * out['power_sum'] + 25 * out['odds_balance'] - 12 * out['low_favorite_flag'].astype(int)
    return out.sort_values('match_score', ascending=False)


def select_match_line(enriched: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    selected = []
    low_favorite_count = 0
    for row in enriched.to_dict('records'):
        if row['low_favorite_flag'] and low_favorite_count >= RULES['max_low_favorite_matches']:
            continue
        selected.append(row)
        low_favorite_count += int(row['low_favorite_flag'])
        if len(selected) == n:
            break
    if len(selected) < n:
        raise ValueError(f'Only selected {len(selected)} matches; need {n}. Add more fixtures or relax constraints.')
    return pd.DataFrame(selected)


enriched_fixtures = enrich_fixtures(fixtures, power)
match_line = select_match_line(enriched_fixtures, RULES['line_matches'])
match_line[['match_id', 'league', 'home_team', 'away_team', 'favorite_odds', 'low_favorite_flag', 'match_score']]

In [ ]:
def build_player_template(match_line: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'bet_no': range(1, RULES['bets_count'] + 1),
        'bet_type': ['single', 'single', 'single', 'single', 'express'],
        'stake': [500, 600, 1000, 1200, 1700],
        'event_1_match_id': [''] * RULES['bets_count'],
        'event_1_market': [''] * RULES['bets_count'],
        'event_2_match_id': [''] * RULES['bets_count'],
        'event_2_market': [''] * RULES['bets_count'],
        'event_3_match_id': [''] * RULES['bets_count'],
        'event_3_market': [''] * RULES['bets_count'],
    })


template = build_player_template(match_line)
template

In [ ]:
MARKET_RE = r'(?:1Х|Х2|П1|П2|ТМ|ТБ|Х)'
STAKE_RE = r'(?:за|на)\s*(\d{3,4})\b|\b(\d{3,4})\s*(?:у\.?е\.?|$)'


def split_events(raw: str) -> list[str]:
    raw = re.sub(r'^(?:\d+[\.)]\s*)?(?:пресс|экспресс)\s*[:\-]?', '', raw.strip(), flags=re.I)
    return [part.strip(' .;') for part in re.split(r'\s*\*\s*', raw) if part.strip(' .;')]


def parse_event(event_text: str) -> dict:
    match = re.search(MARKET_RE, event_text)
    if not match:
        return {'match_text': event_text.strip(), 'market': None}
    market = match.group(0)
    match_text = (event_text[:match.start()] + event_text[match.end():]).strip(' -–—.,;')
    return {'match_text': match_text, 'market': market}


def extract_stake(line: str) -> int | None:
    matches = re.findall(STAKE_RE, line, flags=re.I)
    values = [int(a or b) for a, b in matches]
    return values[-1] if values else None


def parse_player_bets(text: str, player: str = 'unknown') -> pd.DataFrame:
    rows = []
    bet_no = 0
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        stake = extract_stake(line)
        if stake is None:
            continue
        bet_no += 1
        line_without_stake = re.sub(STAKE_RE, '', line, flags=re.I).strip(' .;')
        bet_type = 'express' if re.search(r'пресс|экспресс|\*', line_without_stake, flags=re.I) else 'single'
        events = split_events(line_without_stake) if bet_type == 'express' else [re.sub(r'^\d+[\.)]\s*', '', line_without_stake)]
        for event_no, event in enumerate(events, start=1):
            parsed = parse_event(event)
            rows.append({
                'player': player,
                'bet_no': bet_no,
                'bet_type': bet_type,
                'stake': stake,
                'event_no': event_no,
                **parsed,
            })
    return pd.DataFrame(rows)


sample_text = '''
1. Локомотив-Спартак X за 500.
2. Краснодар-Зенит ТБ за 600.
3. МЮ-Тоттенхэм П1 за 1000.
4. пресс. Атлетико М-Реал X2*Бавария-Байер П1*Сельта-Жирона 1Х за 1700
5. пресс Тулуза-Лион ТБ*Торино-Лацио X2 за 1200.
'''

parsed_bets = parse_player_bets(sample_text, player='demo')
parsed_bets

In [ ]:
def validate_player_bets(parsed: pd.DataFrame) -> pd.DataFrame:
    issues = []
    if parsed.empty:
        return pd.DataFrame([{'level': 'error', 'issue': 'Нет распознанных ставок'}])

    bets = parsed[['bet_no', 'bet_type', 'stake']].drop_duplicates()
    stake_sum = bets['stake'].sum()
    if len(bets) != RULES['bets_count']:
        issues.append({'level': 'error', 'issue': f'Ставок {len(bets)}, нужно {RULES["bets_count"]}'})
    if stake_sum != RULES['bank']:
        issues.append({'level': 'error', 'issue': f'Сумма ставок {stake_sum}, нужно {RULES["bank"]}'})
    bad_stakes = bets[(bets['stake'] < RULES['stake_min']) | (bets['stake'] > RULES['stake_max']) | (bets['stake'] % RULES['stake_step'] != 0)]
    for bet_no in bad_stakes['bet_no'].tolist():
        issues.append({'level': 'error', 'issue': f'Некорректный размер ставки в bet_no={bet_no}'})

    bad_markets = parsed[~parsed['market'].isin(RULES['allowed_markets'])]
    for _, row in bad_markets.iterrows():
        issues.append({'level': 'error', 'issue': f'Недопустимый рынок: bet_no={row.bet_no}, event_no={row.event_no}, market={row.market}'})

    event_counts = parsed.groupby('bet_no').size()
    singles = bets[bets['bet_type'] == 'single']['bet_no'].tolist()
    expresses = bets[bets['bet_type'] == 'express']['bet_no'].tolist()
    for bet_no in singles:
        if event_counts.get(bet_no, 0) != 1:
            issues.append({'level': 'error', 'issue': f'Ординар bet_no={bet_no} содержит не 1 событие'})
    for bet_no in expresses:
        if event_counts.get(bet_no, 0) not in {2, 3}:
            issues.append({'level': 'error', 'issue': f'Экспресс bet_no={bet_no} должен содержать 2 или 3 события'})
    if (len(singles), len(expresses)) not in {(4, 1), (3, 2)}:
        issues.append({'level': 'error', 'issue': f'Структура {len(singles)} ординаров + {len(expresses)} экспрессов не разрешена'})

    repeated_matches = parsed['match_text'].str.lower().value_counts()
    for match_text, cnt in repeated_matches[repeated_matches > 1].items():
        issues.append({'level': 'error', 'issue': f'Матч повторяется: {match_text} ({cnt} раза)'})

    return pd.DataFrame(issues) if issues else pd.DataFrame([{'level': 'ok', 'issue': 'Ошибок не найдено'}])


validation = validate_player_bets(parsed_bets)
validation

In [ ]:
line_path = OUTPUT_DIR / 'tak_ili_inache_match_line.xlsx'
bets_path = OUTPUT_DIR / 'tak_ili_inache_player_bets.xlsx'

with pd.ExcelWriter(line_path) as writer:
    match_line.to_excel(writer, sheet_name='line_12_matches', index=False)
    enriched_fixtures.to_excel(writer, sheet_name='all_candidates', index=False)
    template.to_excel(writer, sheet_name='player_template', index=False)

with pd.ExcelWriter(bets_path) as writer:
    parsed_bets.to_excel(writer, sheet_name='parsed_bets', index=False)
    validation.to_excel(writer, sheet_name='validation', index=False)

line_path, bets_path